# Lezione 5 — Introduzione alle reti convoluzionali

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 90 minuti**

## Obiettivi di apprendimento di questa lezione

- Capire perché le reti completamente connesse (fully connected) sono inefficienti per le immagini
- Capire come funziona la convoluzione: kernel, stride, padding
- Costruire una rete convoluzionale (CNN) con `nn.Conv2d` e pooling
- Confrontare una rete fully connected con una CNN sulla stessa classificazione MNIST

Questo notebook accompagna il capitolo 05 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**.

## Parte 1 — Perché le immagini richiedono un approccio diverso

Negli ultimi notebook hai costruito reti completamente connesse (fully connected, FC) con `nn.Linear`. Ogni neurone dello strato precedente è connesso a ogni neurone dello strato successivo. Per un'immagine MNIST (28×28 pixel = 784 valori), lo strato nascosto di 128 neuroni ha:

784 × 128 = 100,352 pesi

Per un'immagine a colori da 224×224 pixel (una taglia standard in visione artificiale):

224 × 224 × 3 × 128 = 19,267,584 pesi

Solo il primo strato! Questo è:

1. **Computazionalmente costoso**: molti calcoli, memoria, tempo di addestramento
2. **Innaturale**: un'immagine ha struttura spaziale (pixel vicini sono correlati), e una rete fully connected la ignora
3. **Scarso con dati limitati**: tanti parametri = alto rischio di sovradattamento

Una **convoluzione** risolve questi problemi sfruttando la struttura locale: un piccolo kernel (5×5, 3×3) scorre su l'immagine, riutilizzando gli stessi pesi in ogni posizione.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

print('Versión de PyTorch:', torch.__version__)
print('Versión de torchvision:', torchvision.__version__)

In [ ]:
# Cargamos MNIST con torchvision, igual que en la clase 2, y lo servimos en
# lotes (batches) con DataLoader
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Tomamos un único lote de ejemplo: lo usaremos en el resto del notebook
imagenes, etiquetas = next(iter(train_loader))

print('Forma del lote de imágenes:', imagenes.shape)   # (lote, canales, alto, ancho)
print('Forma del lote de etiquetas:', etiquetas.shape)

plt.imshow(imagenes[0][0], cmap='gray')
plt.title(f'Etiqueta: {etiquetas[0].item()}')
plt.axis('off')
plt.show()

## Parte 2 — Anatomia di una convoluzione

### Il kernel (filtro)

Un kernel è una piccola matrice (per esempio 3×3 o 5×5) di pesi che ricercano pattern locali:

- Un kernel addestrato per rilevare bordi verticali
- Un altro per rilevare angoli
- Un altro ancora per rilevare texture

Il modello impara quali kernel servono ai suoi scopi, proprio come impara gli altri pesi.

### Stride e padding

- **Stride**: di quanti pixel il kernel si muove ad ogni passata. Stride=1 significa muoversi di 1 pixel, stride=2 di 2 pixel. Stride più grande = output più piccolo.
- **Padding**: quanti zeri aggiungere intorno all'immagine prima della convoluzione. Padding=1 significa aggiungere una riga/colonna di zeri. Serve per preservare la dimensione spaziale.

### L'operazione

Ad ogni posizione, il kernel esegue una moltiplicazione elemento per elemento con la regione sottostante e somma il risultato: come un prodotto scalare localizzato.

In [ ]:
def aplicar_filtro(imagen, filtro):
    """Aplica manualmente un filtro de convolución 3x3 (stride=1, sin padding)
    a una imagen en escala de grises representada como array 2D de numpy."""
    imagen_filtrada = np.zeros_like(imagen)
    alto, ancho = imagen.shape

    for x in range(1, alto - 1):
        for y in range(1, ancho - 1):
            suma = 0.0
            for fx in range(3):
                for fy in range(3):
                    suma += imagen[x + fx - 1, y + fy - 1] * filtro[fx][fy]
            imagen_filtrada[x, y] = suma

    return imagen_filtrada


imagen_np = imagenes[0][0].numpy()

filtro_horizontal = [[-1, -2, -1],
                      [ 0,  0,  0],
                      [ 1,  2,  1]]

imagen_filtrada = aplicar_filtro(imagen_np, filtro_horizontal)

fig, ejes = plt.subplots(1, 2, figsize=(6, 3))
ejes[0].imshow(imagen_np, cmap='gray')
ejes[0].set_title('Original')
ejes[0].axis('off')
ejes[1].imshow(imagen_filtrada, cmap='gray')
ejes[1].set_title('Tras el filtro (bordes horizontales)')
ejes[1].axis('off')
plt.show()

## Parte 3 — Pooling

Dopo la convoluzione, spesso si usa un'operazione di **pooling** per ridurre la dimensione spaziale e estrarre le caratteristiche più importanti. Il **Max Pooling** prende il valore massimo in ogni finestra (per es. 2×2):

Un'immagine 4×4 diventa 2×2 dopo Max Pooling 2×2. Questo non solo riduce la dimensione (meno parametri nel prossimo strato), ma aiuta anche a rilevare se una caratteristica è presente in una regione, indipendentemente dalla sua posizione esatta.

In [ ]:
def max_pooling(imagen, tamano=2):
    """Aplica max pooling de tamano x tamano sobre una imagen en escala de grises."""
    alto, ancho = imagen.shape
    nueva_alto, nueva_ancho = alto // tamano, ancho // tamano
    imagen_reducida = np.zeros((nueva_alto, nueva_ancho))

    for x in range(0, alto - 1, tamano):
        for y in range(0, ancho - 1, tamano):
            bloque = imagen[x:x + tamano, y:y + tamano]
            imagen_reducida[x // tamano, y // tamano] = np.max(bloque)

    return imagen_reducida


imagen_pooled = max_pooling(imagen_filtrada, tamano=2)

print('Tamaño antes del pooling:', imagen_filtrada.shape)
print('Tamaño después del pooling:', imagen_pooled.shape)

plt.imshow(imagen_pooled, cmap='gray')
plt.title('Después de max pooling 2x2')
plt.axis('off')
plt.show()

## Parte 4 — Architettura di una CNN

Una CNN tipica per la classificazione è:

```
Input (immagine) 
  → Conv2d (in_channels, out_channels, kernel_size) 
  → ReLU 
  → MaxPool2d 
  → Conv2d 
  → ReLU 
  → MaxPool2d 
  → Flatten 
  → Linear (fully connected per la classificazione)
```

I primi strati imparano feature di basso livello (bordi, texture), i strati intermedi combinano quelle in feature di livello medio (forme), e gli ultimi strati le usano per classificare.

In [ ]:
class CNNSimple(nn.Module):
    """CNN sencilla para clasificar MNIST (1 canal, 28x28 píxeles) en 10 clases."""

    def __init__(self):
        super().__init__()
        # Primer bloque: 1 canal de entrada (escala de grises), 16 filtros de
        # 3x3, con padding=1 para no perder tamaño en los bordes
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3,
                                stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Segundo bloque: recibe los 16 mapas de características del primer
        # bloque y aprende 32 filtros nuevos, combinando esos rasgos
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,
                                stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Capa final densa: clasifica a partir de los mapas de características
        # ya reducidos por las dos capas de pooling (28 -> 14 -> 7)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)   # aplanamos justo antes de la capa densa
        x = self.fc(x)
        return x


modelo = CNNSimple()
print(modelo)

In [ ]:
# Forward pass sobre el lote real de MNIST cargado en la Parte 1,
# imprimiendo la forma del tensor después de cada capa
x = imagenes
print('Entrada:                ', x.shape)

x = F.relu(modelo.conv1(x))
print('Tras conv1 + ReLU:       ', x.shape)

x = modelo.pool1(x)
print('Tras pool1:              ', x.shape)

x = F.relu(modelo.conv2(x))
print('Tras conv2 + ReLU:       ', x.shape)

x = modelo.pool2(x)
print('Tras pool2:              ', x.shape)

x = x.view(x.size(0), -1)
print('Tras aplanar (flatten):  ', x.shape)

x = modelo.fc(x)
print('Tras fc (salida):        ', x.shape)

## Parte 5 — Implementare una CNN in PyTorch

`nn.Conv2d` è lo strato di convoluzione di PyTorch:

```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

Per MNIST (immagini in scala di grigi, 1 canale):

```python
nn.Conv2d(1, 32, kernel_size=3, padding=1)
```

questo crea 32 kernel 3×3 che ricercano pattern nell'immagine.

`nn.MaxPool2d` è il pooling:

```python
nn.MaxPool2d(kernel_size=2, stride=2)
```

questo riduce l'immagine 2×2 → 1×1, prendendo il massimo.